# 弱智吧AI · ruozhi

参考 [karpathy/nanochat](https://github.com/karpathy/nanochat)，在 Colab 上从零训练一个会讲弱智吧金句的小语言模型：

1. **数据**：GitHub / HuggingFace 上开源的百度弱智吧数据集 → SFT 对话数据
2. **分词器**：在中文网页 + 弱智吧语料上训练 16k 词表的 byte-level BPE
3. **预训练**：FineWeb-2 中文（通用网页）+ 少量弱智吧原文
4. **SFT**：弱智吧问答 / 金句 / 续写
5. **聊天**：命令行或 Gradio 网页（可生成公开链接）

**推荐分两段跑**（数据准备只用 CPU，不必占着 A100）：

1. **CPU 运行时**：打开第 2 节的 `USE_DRIVE`，`运行时 → 全部运行`。第 1–4 节会下载数据、训练分词器、编码 token 文件，并备份到 Google Drive；到第 5 节检测到没有 GPU 会报错停下，这是预期的。编码很吃 CPU，核数越多越快。
2. **A100 运行时**：`运行时 → 断开连接并删除运行时`，切换到 A100，再`全部运行`。已准备好的数据会从 Drive 恢复，第 3–4 节自动跳过，直接开始训练。

默认 d8（4200 万参数）：数据准备约 40 分钟（CPU），A100 上预训练 30–50 分钟 + SFT 几分钟。只用 T4 的话建议把 `DEPTH` 改成 6。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "没有 GPU：只能跑第 1–4 节（数据准备）"
!nproc

## 1. 获取代码

In [ ]:
REPO = "https://github.com/ubisoft-potato/ruozhi.git"
BRANCH = "main"  # @param {type:"string"}
!rm -rf /content/ruozhi && git clone -q -b {BRANCH} {REPO} /content/ruozhi
%cd /content/ruozhi
!pip install -q tokenizers datasets gradio

## 2. （可选）用 Google Drive 保存进度

Colab 断线会清空 `/content`。打开 `USE_DRIVE` 后，启动时会从 Drive 恢复已有的数据 / 分词器 / 检查点；数据准备完会把 token 文件备份到 Drive，最后一格备份检查点。
**CPU / GPU 分两段跑时必须打开。** 训练本身读写本地磁盘（直接读 Drive 上的 memmap 很慢）。

In [ ]:
USE_DRIVE = True  # @param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/ruozhi_artifacts"
import os
os.environ["RUOZHI_BASE_DIR"] = "/content/ruozhi/artifacts"
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if os.path.exists(DRIVE_DIR):
        !mkdir -p /content/ruozhi/artifacts && rsync -a {DRIVE_DIR}/ /content/ruozhi/artifacts/
        !ls -R /content/ruozhi/artifacts | head -40

## 3. 弱智吧数据 → SFT 对话

In [ ]:
import os
if os.path.exists("artifacts/sft/train.jsonl") and os.path.exists("artifacts/ruozhiba_corpus.jsonl"):
    print("已有弱智吧数据（从 Drive 恢复），跳过")
else:
    !python -m scripts.prepare_ruozhiba
!head -n 5 artifacts/sft/train.jsonl

## 4. 预训练数据 + 分词器

从 HuggingFace 流式读取 FineWeb-2 中文（`cmn_Hani`），训练分词器并把文本编码成 `uint16` token 文件。

`DEPTH` 在这里先选好：`MAX_TOKENS` 会按它自动算成训练所需 token（20 × 参数量）的 60%，数据最多重复约 2 遍，下限 3 亿。
编码的瓶颈是 CPU，3 亿 token 约 20–30 分钟，d8 的 5 亿约 35–50 分钟（A100 运行时 CPU 核数多，会快一些）。只想快速跑通可以把 `MAX_TOKENS` 直接改成 `30_000_000`。
已有足够的 token 文件时（比如从 Drive 恢复）会跳过这一步；想重建就勾选 `REBUILD`（同时会重训分词器，之前的检查点作废）。

In [ ]:
DEPTH = 8  # @param {type:"integer"}
DATASET = "fineweb2"  # @param ["fineweb2", "fineweb-edu-zh", "wiki"]
REBUILD = False  # @param {type:"boolean"}
import json, os
# 参数量 ≈ 12·d·(64d)² + 2·vocab·64d（d8 ≈ 42M）
NUM_PARAMS = 12 * DEPTH * (64 * DEPTH) ** 2 + 2 * 16384 * 64 * DEPTH
MAX_TOKENS = int(max(300e6, 0.6 * 20 * NUM_PARAMS))
print(f"depth {DEPTH}: ~{NUM_PARAMS / 1e6:.0f}M params, train on {20 * NUM_PARAMS / 1e6:.0f}M tokens, need {MAX_TOKENS / 1e6:.0f}M in train.bin")

meta_path = "artifacts/pretrain/meta.json"
have = json.load(open(meta_path)) if os.path.exists(meta_path) and os.path.exists("artifacts/pretrain/train.bin") else None
if have and not REBUILD and have["web_tokens"] >= 0.95 * MAX_TOKENS and have["dataset"] == DATASET:
    print(f"已有 token 文件：{have['train_tokens'] / 1e6:.0f}M tokens（{have['dataset']}），跳过")
else:
    FORCE = "--force_tokenizer" if REBUILD else ""
    !python -m scripts.prepare_pretrain --max_tokens {MAX_TOKENS} --dataset {DATASET} {FORCE}
    if USE_DRIVE:  # 备份数据，换 A100 运行时后直接恢复
        !mkdir -p {DRIVE_DIR} && rsync -a --info=progress2 /content/ruozhi/artifacts/ {DRIVE_DIR}/
        print("数据已备份到 Drive")

## 5. 预训练

`DEPTH`（上一节设置）是唯一的模型大小旋钮：`n_embd = 64 × depth`。默认按 Chinchilla 比例训练 `20 × 参数量` 个 token。

| depth | 参数量 | 训练 token | T4 | A100 |
|---|---|---|---|---|
| 4 | 11.5M | 230M | ~15 min | ~3–5 min |
| 6 | 23.2M | 464M | ~1 h | ~10–15 min |
| 8（默认） | 42.0M | 840M | ~2–3 h | ~30–50 min |

耗时按 MFU 20–35% 粗估，只含预训练；小模型喂不饱 A100，以日志里的 `mfu` / `eta` 为准。断线后加 `--resume` 可从最近的检查点继续。

每步总 token 固定为 131072，`DEVICE_BATCH_SIZE` 只影响显存和速度（越大梯度累积越少）。设为 0 时按显存自动选：≥40GB 的卡（A100）上 d≤8 用 128、更深用 64，其他卡用 32。OOM 就改小。

In [ ]:
DEVICE_BATCH_SIZE = 0  # @param {type:"integer"}
import torch
if not torch.cuda.is_available():
    raise RuntimeError("数据已准备好。预训练需要 GPU：请切换到 A100 运行时后重新「全部运行」")
if DEVICE_BATCH_SIZE <= 0:
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 2**30
    DEVICE_BATCH_SIZE = (128 if DEPTH <= 8 else 64) if gpu_mem >= 39 else 32
print(f"device_batch_size = {DEVICE_BATCH_SIZE}")
!python -m scripts.base_train --depth {DEPTH} --device_batch_size {DEVICE_BATCH_SIZE}

## 6. SFT：学习弱智吧

In [ ]:
!python -m scripts.sft_train --depth {DEPTH}

## 7. 试一试

In [ ]:
for p in ["来一条弱智吧金句", "只剩一个心脏了还能活吗？", "为什么我爸妈结婚的时候没有邀请我？", "你是谁？"]:
    print("你>", p)
    !python -m scripts.chat_cli --depth {DEPTH} -p "{p}"
    print()

网页版（`--share` 会给出一个 72 小时有效的公开链接，可以发给朋友）。停止运行这一格即关闭。

In [ ]:
!python -m scripts.chat_web --depth {DEPTH} --share

## 8. 备份到 Drive

In [ ]:
if USE_DRIVE:
    # 全量同步；Drive 上已有且未变的文件（如 token 文件）rsync 会跳过
    !mkdir -p {DRIVE_DIR} && rsync -a /content/ruozhi/artifacts/ {DRIVE_DIR}/

## 9. （进阶）Scaling law 实验

对每个计算量预算 C、每个 depth 都用恰好 C FLOPs 训练，画出 IsoFLOP 曲线，拟合最优参数量 N\*(C) 和 token 数 D\*(C)，用来决定下一步把模型扩到多大。
T4 上默认配置约 1.5 小时。

In [ ]:
!python -m scripts.scaling_laws --budgets 1e15 3e15 1e16 --depths 2 3 4 5 6 8 --extra_args "--eval_tokens 524288"
from IPython.display import Image
Image("artifacts/scaling/scaling_laws.png")